# 08 · Tier 3 — Modern / senior-depth

Examine **Tier 3** models and eval utilities:

| Component | Module |
|---|---|
| HSTU (time-aware sequential) | `models/hstu.py` |
| Semantic ID LM | `models/semantic_ids.py` |
| Contrastive two-tower | `models/contrastive.py` |
| MMR / DPP slate diversity | `eval/slate.py` |
| IPS debiased metrics | `eval/debias.py` |

Uses the same cross-regime split as `scripts/benchmark.py`.

In [ ]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from src.recsys.config import settings
from src.recsys.data import load_dataset
from src.recsys.eval import evaluate_ips, item_propensity
from src.recsys.eval.slate import diversify_slate
from src.recsys.models import (
    ContrastiveTwoTowerRecommender,
    HSTURecommender,
    ItemTokenLMRecommender,
    SASRecRecommender,
    SemanticIDRecommender,
    TwoTowerRecommender,
)
from src.recsys.models.semantic_ids import build_semantic_item_tokens
from src.recsys.models.text_embeddings import encode_item_text
from scripts.benchmark import (
    _score_recs,
    build_slices,
    build_unified_two_stage,
    slice_ground_truth,
)

ds = load_dataset()
train, test_pos, cold_items, cold_users = build_slices(ds, cutoff_quantile=0.9)
warm_gt, ci_gt, cu_gt = slice_ground_truth(test_pos, cold_items, cold_users)
users = sorted(set(warm_gt) | set(ci_gt) | set(cu_gt))
k = settings.top_k
propensity = item_propensity(train)

print(ds.summary())

## 1. HSTU vs SASRec (sequential warm-user signal)

HSTU adds bucketed **time-delta** embeddings between actions; same causal sequential role as SASRec.

In [ ]:
SEQ = dict(dim=64, epochs=6, max_len=50)

seq_rows = []
for name, cls, kwargs in [
    ("sasrec", SASRecRecommender, SEQ),
    ("hstu", HSTURecommender, {**SEQ, "epochs": 6}),
]:
    model = cls(**kwargs)
    model.fit(train)
    recs = model.recommend(users, k=k)
    m = _score_recs(recs, test_pos, warm_gt, ci_gt, cu_gt, k)
    ips = evaluate_ips(recs, test_pos, propensity, k=k)
    seq_rows.append({"model": name, **m, **ips})

seq_df = pd.DataFrame(seq_rows).set_index("model")
seq_df

## 2. Semantic ID LM vs flat item-token LM

Semantic IDs compose tokens from **category + cluster + leaf** — shared prefixes for similar items.

In [ ]:
item_to_tokens, triplet_map = build_semantic_item_tokens(ds.items, n_clusters=32)
print(f"Semantic vocabulary: {len({t for toks in item_to_tokens.values() for t in toks})} tokens")
print(f"Example item → tokens: {list(item_to_tokens.items())[:2]}")

In [ ]:
gen_rows = []
for name, model in [
    ("item_token_lm", ItemTokenLMRecommender(dim=64, epochs=6, max_len=50)),
    ("semantic_id_lm", SemanticIDRecommender(dim=64, epochs=6, n_clusters=32)),
]:
    if name == "semantic_id_lm":
        model.fit(train, items=ds.items)
    else:
        model.fit(train)
    recs = model.recommend(users, k=k)
    m = _score_recs(recs, test_pos, warm_gt, ci_gt, cu_gt, k)
    gen_rows.append({"model": name, **m})

pd.DataFrame(gen_rows).set_index("model")

## 3. Contrastive two-tower vs baseline

Hard negatives sampled from popularity; optional embedding dropout during training.

In [ ]:
TT = dict(dim=64, epochs=6)
tt_rows = []
for name, model in [
    ("two_tower", TwoTowerRecommender(**TT)),
    ("contrastive_two_tower", ContrastiveTwoTowerRecommender(**TT, hard_negative_k=5)),
]:
    model.fit(train)
    recs = model.recommend(users, k=k)
    tt_rows.append({"model": name, **_score_recs(recs, test_pos, warm_gt, ci_gt, cu_gt, k)})

pd.DataFrame(tt_rows).set_index("model")

## 4. Slate diversity (MMR)

Post-rank a long candidate list for **relevance–diversity** tradeoff. Compare raw vs MMR on unified pipeline.

In [ ]:
emb, item_ids, _ = encode_item_text(ds.items)
item_emb = {iid: emb[i] for i, iid in enumerate(item_ids)}

unified = build_unified_two_stage(ds, use_sasrec=False)
unified.fit(train)
long_recs = unified.recommend(users, k=k * 5)

lambdas = [1.0, 0.7, 0.4]
div_rows = []
for lam in lambdas:
    if lam >= 1.0:
        recs = {u: items[:k] for u, items in long_recs.items()}
        label = "raw (λ=1)"
    else:
        recs = diversify_slate(long_recs, item_emb, k=k, method="mmr", lambda_=lam)
        label = f"mmr λ={lam}"
    m = _score_recs(recs, test_pos, warm_gt, ci_gt, cu_gt, k)
    from src.recsys.eval import coverage
    cov = coverage(recs, ds.n_items, k)
    div_rows.append({"variant": label, "coverage@k": cov, **m})

div_df = pd.DataFrame(div_rows).set_index("variant")
div_df

In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(div_df.index, div_df[f"overall_r@{k}"], "o-", label=f"recall@{k}")
ax1.set_ylabel("recall"); ax1.tick_params(axis="x", rotation=15)
ax2 = ax1.twinx()
ax2.plot(div_df.index, div_df["coverage@k"], "s--", color="C1", label="coverage")
ax2.set_ylabel("catalog coverage")
plt.title("Relevance–diversity Pareto (MMR on unified recs)")
fig.tight_layout()

## 5. IPS debiased evaluation

Down-weight popular items in offline metrics. Compare raw vs IPS recall on unified recommendations.

In [ ]:
raw_recs = {u: items[:k] for u, items in long_recs.items()}
ips_overall = evaluate_ips(raw_recs, test_pos, propensity, k=k)
ips_warm = evaluate_ips(raw_recs, warm_gt, propensity, k=k)
ips_cold_user = evaluate_ips(raw_recs, cu_gt, propensity, k=k)

ips_df = pd.DataFrame([
    {"slice": "overall", **ips_overall},
    {"slice": "warm", **ips_warm},
    {"slice": "cold_user", **ips_cold_user},
]).set_index("slice")
ips_df

## 6. Tier 3 summary table

Combine sequential, generative, and contrastive runs from above.

In [ ]:
summary = pd.concat([seq_df, pd.DataFrame(gen_rows).set_index("model"), pd.DataFrame(tt_rows).set_index("model")])
cols = [f"overall_r@{k}", f"warm_r@{k}", f"cold_item_r@{k}", f"cold_user_r@{k}"]
summary[cols].plot.bar(figsize=(12, 4), rot=0, title="Tier 3 models — four-view recall")
plt.ylabel(f"recall@{k}"); plt.legend(bbox_to_anchor=(1.02, 1)); plt.tight_layout()

## Next steps

- Full Tier 3 benchmark: `python scripts/benchmark.py --mode tier3`
- With diversity + IPS: `python scripts/benchmark.py --only two_stage_unified --diversify --ips`
- Interview guide: `docs/techniques.md` § Senior / Staff MLE analyses